In [0]:
# PySpark SQL Functions

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    isnan,
    initcap,
    desc,
    asc,
    avg,
    sum,
    min,
    max,
    round,
    trim,
    lower,
    upper,
    split,
    explode,
    year,
    month,
    quarter,
    weekofyear,
    dayofmonth,
    datediff,
    current_date,
    lit,
    regexp_replace
)


# Window Functions

from pyspark.sql.window import Window


# PySpark Data Types

from pyspark.sql.types import (
    StringType,
    IntegerType,
    LongType,
    DateType,
    DoubleType
)

In [0]:
base_path = "/Volumes/project/datasets/spotify/"
bronze_song_charts = spark.read.parquet(
    "/Volumes/project/datasets/spotify/charts_songs_daily.parquet"
)
print("Rows:", bronze_song_charts.count())

print("Columns:", len(bronze_song_charts.columns))

Rows: 42757018
Columns: 18


In [0]:
silver_song_charts = bronze_song_charts

In [0]:
# Remove Duplicate Chart Events
silver_song_charts = (
    silver_song_charts
    .dropDuplicates(["date","country","uri"])
)

print("Rows:", silver_song_charts.count())

Rows: 42756826


In [0]:
# Handle Null Values
silver_song_charts = (
    silver_song_charts
    .fillna({
        "artist_names": "Unknown Artist",
        "track_name": "Unknown Track",
        "label": "Independent/Unknown"
    })
)

In [0]:
validation_check = (
    silver_song_charts
    .select(
        count(when(~col("rank").between(1,200), True)).alias("invalid_rank_records"),
        count(when(col("streams") <= 0, True)).alias("invalid_stream_records"),
        min("rank").alias("minimum_rank"),
        max("rank").alias("maximum_rank"),
        min("streams").alias("minimum_streams"),
        max("streams").alias("maximum_streams")
    )
)

validation_check.show()

+--------------------+----------------------+------------+------------+---------------+---------------+
|invalid_rank_records|invalid_stream_records|minimum_rank|maximum_rank|minimum_streams|maximum_streams|
+--------------------+----------------------+------------+------------+---------------+---------------+
|                   0|                     0|           1|         200|           1001|       30987370|
+--------------------+----------------------+------------+------------+---------------+---------------+



No invalid records

Ranks between 1 - 200

No -ve streams

Hence no transformation required here

In [0]:
text_quality_check = (
    silver_song_charts
    .select(
        count(when(col("country") != trim(col("country")), True)).alias("country_spaces"),
        count(when(col("track_name") != trim(col("track_name")), True)).alias("track_spaces"),
        count(when(col("artist_names") != trim(col("artist_names")), True)).alias("artist_spaces"),
        count(when(col("label") != trim(col("label")), True)).alias("label_spaces"),
        count(when(trim(col("track_name")) == "", True)).alias("empty_tracks"),
        count(when(trim(col("artist_names")) == "", True)).alias("empty_artists"),
        count(when(trim(col("label")) == "", True)).alias("empty_labels")
    )
)

text_quality_check.show()

+--------------+------------+-------------+------------+------------+-------------+------------+
|country_spaces|track_spaces|artist_spaces|label_spaces|empty_tracks|empty_artists|empty_labels|
+--------------+------------+-------------+------------+------------+-------------+------------+
|             0|           0|         5076|           0|           0|            0|           0|
+--------------+------------+-------------+------------+------------+-------------+------------+



In [0]:
artist_separator_check = (
    silver_song_charts
    .select(
        count(when(col("artist_names").contains(","), True)).alias("comma_separator"),
        count(when(col("artist_names").contains("|"), True)).alias("pipe_separator"),
        count(when(col("artist_names").contains(";"), True)).alias("semicolon_separator")
    )
)

artist_separator_check.show()

+---------------+--------------+-------------------+
|comma_separator|pipe_separator|semicolon_separator|
+---------------+--------------+-------------------+
|          60714|      17666231|                  3|
+---------------+--------------+-------------------+



In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn("artist_names", trim(col("artist_names")))
)

In [0]:
# Create Artist Mapping Table
from pyspark.sql.functions import arrays_zip

silver_artist_mapping = (
    silver_song_charts
    .select(
        "uri",
        explode(
            arrays_zip(
                split(col("artist_uris"), "\\|"),
                split(col("artist_names"), "\\|")
            )
        ).alias("artist")
    )
    .select(
        col("uri"),
        trim(col("artist.0")).alias("artist_uri"),
        trim(col("artist.1")).alias("artist_name")
    )
    .dropDuplicates()
)

In [0]:
# Validate Mapping
silver_artist_mapping.show(10, False)

silver_artist_mapping.select(
    countDistinct("artist_uri").alias("unique_artists")
).show()

print("Rows : ", silver_artist_mapping.count())

+------------------------------------+-------------------------------------+-------------+
|uri                                 |artist_uri                           |artist_name  |
+------------------------------------+-------------------------------------+-------------+
|spotify:track:7szuecWAPwGoV1e5vGu8tl|spotify:artist:1Xyo4u8uXC1ZmMpatF05PJ|The Weeknd   |
|spotify:track:100fbGocyqbBPQkTDchi2g|spotify:artist:6HaGTQPmzraVmaVxvz6EUc|Jung Kook    |
|spotify:track:6RyaV7owmVU6fzEPE17sF1|spotify:artist:4VMYDCV2IEDYJArk749S6m|Daddy Yankee |
|spotify:track:52XYwQKlXp7scE7KrBBCID|spotify:artist:6wWVKhxIU2cEi0K81v7HvP|Rammstein    |
|spotify:track:3swc6WTsr7rl9DqQKQA55C|spotify:artist:7c0XG5cIJTrrAgEC3ULPiq|Ty Dolla $ign|
|spotify:track:2eAvDnpXP5W0cVtiI0PUxV|spotify:artist:2WzaAvm2bBCf4pEhyuDgCY|Ruth B.      |
|spotify:track:6DNtNfH8hXkqOX1sjqmI7p|spotify:artist:0bdfiayQAKewqEvaU6rXCv|MØ           |
|spotify:track:35cOyocq8Gb6UcT0NWeTwn|spotify:artist:4RfOLIFy2xEmlWzXEVmLJn|Sjava        |

In [0]:
collaboration_check = (
    silver_artist_mapping
    .select(
        count(when(col("artist_name").contains("|"), True)).alias("pipe_remaining"),
        count(when(col("artist_name").contains(";"), True)).alias("semicolon_remaining")
    )
)

collaboration_check.show()

+--------------+-------------------+
|pipe_remaining|semicolon_remaining|
+--------------+-------------------+
|             0|                  2|
+--------------+-------------------+



In [0]:
multi_artist_tracks = (
    silver_artist_mapping
    .groupBy("uri")
    .agg(countDistinct("artist_uri").alias("artist_count"))
    .filter(col("artist_count") > 1)
)

multi_artist_tracks.show(10, False)

+------------------------------------+------------+
|uri                                 |artist_count|
+------------------------------------+------------+
|spotify:track:59eeMIDGp3dDS0c3CcndO4|3           |
|spotify:track:1rP330OiugE0As2HUbTJG9|3           |
|spotify:track:6Rz80wUWSUbGGVHvkH7NpU|2           |
|spotify:track:0iMGyvMJqv21SF6iTxrcvQ|3           |
|spotify:track:7ndejYcYfmvkbImaXiYf9C|3           |
|spotify:track:62M1Eus2vfIGmtq6zJ1yHI|3           |
|spotify:track:1HsCW2LNTPxDa5khZbBNxW|2           |
|spotify:track:2mM3gZ0BbPwPPMelbA8vgt|5           |
|spotify:track:4Cy5f4JsH1yoeGks7FnoHw|3           |
|spotify:track:40MgoUXdjPRk27R1Fp7EAp|2           |
+------------------------------------+------------+
only showing top 10 rows


In [0]:
artist_per_song_distribution = (
    silver_artist_mapping
    .groupBy("uri")
    .agg(countDistinct("artist_uri").alias("artist_count"))
    .groupBy("artist_count")
    .count()
    .orderBy("artist_count")
)

artist_per_song_distribution.show()

+------------+------+
|artist_count| count|
+------------+------+
|           1|150543|
|           2| 63002|
|           3| 19013|
|           4|  5504|
|           5|  1913|
|           6|   855|
|           7|   425|
|           8|   214|
|           9|   148|
|          10|    82|
|          11|    54|
|          12|    20|
|          13|    16|
|          14|    17|
|          15|    11|
|          16|    10|
|          17|     8|
|          18|     9|
|          19|     4|
|          20|     5|
+------------+------+
only showing top 20 rows


Date Quality Validation

In [0]:
date_quality_check = (
    silver_song_charts
    .select(
        min("date").alias("first_chart_date"),
        max("date").alias("latest_chart_date"),
        min("release_date").alias("oldest_release"),
        max("release_date").alias("latest_release"),
        count(
            when(col("release_date") > col("date"), True)
        ).alias("future_release_errors")
    )
)

date_quality_check.show()

+----------------+-----------------+--------------+--------------+---------------------+
|first_chart_date|latest_chart_date|oldest_release|latest_release|future_release_errors|
+----------------+-----------------+--------------+--------------+---------------------+
|      2017-01-01|       2026-05-20|    1840-06-14|    2026-05-20|              2659828|
+----------------+-----------------+--------------+--------------+---------------------+



Release Date Validation:

Found 26.5M records where release_date appeared after chart date due to Spotify metadata refreshes.

Action:
Original release_date preserved.
Created valid_release_date for analytical calculations.
Invalid future dates converted to NULL.

No chart records removed.

Create a corrected feature : valid_release_date. Keep release_date i.e the original column

In [0]:
release_quality_song_level = (
    silver_song_charts
    .groupBy("uri")
    .agg(
        min("date").alias("first_chart_date"),
        min("release_date").alias("release_date")
    )
    .select(
        count("*").alias("total_songs"),
        count(
            when(
                col("release_date") <= col("first_chart_date"),
                True
            )
        ).alias("valid_release_songs"),
        count(
            when(
                col("release_date") > col("first_chart_date"),
                True
            )
        ).alias("invalid_release_songs")
    )
)

release_quality_song_level.show()

+-----------+-------------------+---------------------+
|total_songs|valid_release_songs|invalid_release_songs|
+-----------+-------------------+---------------------+
|     241874|             216331|                21787|
+-----------+-------------------+---------------------+



In [0]:
# Create Valid Release Date
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "valid_release_date",
        when(
            col("release_date") <= col("date"),
            col("release_date")
        )
    )
)

In [0]:
silver_song_charts.select(
    count(
        when(
            col("valid_release_date") > col("date"),
            True
        )
    ).alias("remaining_release_errors")
).show()

+------------------------+
|remaining_release_errors|
+------------------------+
|                       0|
+------------------------+



Time Intelligence Features

In [0]:
#Create Date Features
silver_song_charts = (
    silver_song_charts
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("week", weekofyear(col("date")))
)

In [0]:
# Create Song Age Feature
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "song_age_days",
        datediff(
            col("date"),
            col("valid_release_date")
        )
    )
)

In [0]:
# Create Song Age Category
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "song_age_category",
        when(
            col("song_age_days") <= 90,
            "New Release"
        )
        .when(
            col("song_age_days") <= 365,
            "Recent Hit"
        )
        .when(
            col("song_age_days") <= 1825,
            "Established"
        )
        .when(
            col("song_age_days") > 1825,
            "Evergreen"
        )
        .otherwise("Unknown")
    )
)

Validate Song Age Category Distribution

In [0]:
song_age_distribution = (
    silver_song_charts
    .groupBy("song_age_category")
    .agg(
        count("*").alias("records"),
        countDistinct("uri").alias("unique_songs"),
        sum("streams").alias("total_streams")
    )
    .orderBy(desc("records"))
)

song_age_distribution.show(20, False)

+-----------------+--------+------------+-------------+
|song_age_category|records |unique_songs|total_streams|
+-----------------+--------+------------+-------------+
|Recent Hit       |12991896|52897       |840336307109 |
|New Release      |11564051|177223      |977064601556 |
|Established      |8805029 |32578       |440265255190 |
|Evergreen        |6099774 |24514       |408422994766 |
|Unknown          |3296076 |25596       |232823611893 |
+-----------------+--------+------------+-------------+



Created song lifecycle segmentation based on release age.

Categories:
- New Release
- Recent Hit
- Established
- Evergreen

Validation showed 89%+ song coverage with reliable release dates, enabling lifecycle-based music intelligence.

In [0]:
silver_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = false)
 |-- track_name: string (nullable = false)
 |-- label: string (nullable = false)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- valid_release_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- song_age_days: integer (nullable = true)
 |-- song_age_category: s

First Validate Rank Movement Logic

In [0]:
rank_quality_check = (
    silver_song_charts
    .select(
        min("rank").alias("min_rank"),
        max("rank").alias("max_rank"),
        min("previous_rank").alias("min_previous_rank"),
        max("previous_rank").alias("max_previous_rank"),
        min("peak_rank").alias("min_peak_rank"),
        max("peak_rank").alias("max_peak_rank")
    )
)

rank_quality_check.show()

+--------+--------+-----------------+-----------------+-------------+-------------+
|min_rank|max_rank|min_previous_rank|max_previous_rank|min_peak_rank|max_peak_rank|
+--------+--------+-----------------+-----------------+-------------+-------------+
|       1|     200|               -1|              200|            1|          200|
+--------+--------+-----------------+-----------------+-------------+-------------+



Create Rank Movement Feature : if prev rank>0 then else dont. As prev rank = -1 means new entry

In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "rank_movement",
        when(
            col("previous_rank") > 0,
            col("previous_rank") - col("rank")
        )
    )
)

In [0]:
movement_validation = (
    silver_song_charts
    .select(
        min("rank_movement").alias("biggest_drop"),
        max("rank_movement").alias("biggest_gain"),
        count(
            when(
                col("rank_movement").isNull(),
                True
            )
        ).alias("new_entries")
    )
)

movement_validation.show()

+------------+------------+-----------+
|biggest_drop|biggest_gain|new_entries|
+------------+------------+-----------+
|        -197|         199|    2743140|
+------------+------------+-----------+



Movement Category

In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "movement_category",
        when(
            col("rank_movement").isNull(),
            "New Entry"
        )
        .when(
            col("rank_movement") >= 50,
            "Strong Gainer"
        )
        .when(
            col("rank_movement") > 0,
            "Gainer"
        )
        .when(
            col("rank_movement") == 0,
            "Stable"
        )
        .when(
            col("rank_movement") <= -50,
            "Strong Decliner"
        )
        .otherwise("Decliner")
    )
)

In [0]:
movement_distribution = (
    silver_song_charts
    .groupBy("movement_category")
    .agg(
        count("*").alias("records"),
        countDistinct("uri").alias("unique_songs")
    )
    .orderBy(desc("records"))
)

movement_distribution.show()

+-----------------+--------+------------+
|movement_category| records|unique_songs|
+-----------------+--------+------------+
|         Decliner|18572118|      168429|
|           Gainer|16558085|      154970|
|           Stable| 4237126|       91192|
|        New Entry| 2743140|      236445|
|  Strong Decliner|  347497|       85146|
|    Strong Gainer|  298860|       63304|
+-----------------+--------+------------+



Hit Tier / Popularity Classification

In [0]:
rank_distribution = (
    silver_song_charts
    .select(
        count(when(col("rank") <= 10, True)).alias("top_10_records"),
        count(when((col("rank") > 10) & (col("rank") <= 50), True)).alias("top_50_records"),
        count(when((col("rank") > 50) & (col("rank") <= 100), True)).alias("top_100_records"),
        count(when(col("rank") > 100, True)).alias("remaining_records")
    )
)

rank_distribution.show()

+--------------+--------------+---------------+-----------------+
|top_10_records|top_50_records|top_100_records|remaining_records|
+--------------+--------------+---------------+-----------------+
|       2269990|       8992231|       10876724|         20617881|
+--------------+--------------+---------------+-----------------+



Create Hit Category

In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "hit_category",
        when(col("rank") <= 10, "Global Hit")
        .when(col("rank") <= 50, "Major Hit")
        .when(col("rank") <= 100, "Popular Track")
        .otherwise("Charting Track")
    )
)

In [0]:
hit_category_check = (
    silver_song_charts
    .groupBy("hit_category")
    .agg(
        count("*").alias("records"),
        countDistinct("uri").alias("unique_songs"),
        sum("streams").alias("total_streams")
    )
    .orderBy(desc("records"))
)

hit_category_check.show()

+--------------+--------+------------+-------------+
|  hit_category| records|unique_songs|total_streams|
+--------------+--------+------------+-------------+
|Charting Track|20617881|      225993| 893828855247|
| Popular Track|10876724|      149570| 646812924427|
|     Major Hit| 8992231|       92299| 894513451564|
|    Global Hit| 2269990|       32323| 463757539276|
+--------------+--------+------------+-------------+



Chart Strength Score

based on:

1. Rank performance
2. Streams
3. Longevity
4. Peak performance

(rank score)
+
(stream score)
+
(longevity score)

In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "chart_strength_score",
        (
            ((201 - col("rank")) * 0.4) +
            ((201 - col("peak_rank")) * 0.3) +
            (col("days_on_chart") * 0.2) +
            (col("consecutive_days") * 0.1)
        )
    )
)

In [0]:
strength_check = (
    silver_song_charts
    .select(
        min("chart_strength_score").alias("minimum_score"),
        max("chart_strength_score").alias("maximum_score"),
        avg("chart_strength_score").alias("average_score")
    )
)

strength_check.show()

+------------------+------------------+-----------------+
|     minimum_score|     maximum_score|    average_score|
+------------------+------------------+-----------------+
|0.9999999999999999|1166.1000000000001|157.2655144911832|
+------------------+------------------+-----------------+



Stream Performance Tier

In [0]:
stream_distribution = (
    silver_song_charts
    .select(
        min("streams").alias("minimum_streams"),
        avg("streams").alias("average_streams"),
        max("streams").alias("maximum_streams")
    )
)

stream_distribution.show()

+---------------+----------------+---------------+
|minimum_streams| average_streams|maximum_streams|
+---------------+----------------+---------------+
|           1001|67799.9992448925|       30987370|
+---------------+----------------+---------------+



Mega Hit        >= 10M daily streams

Super Hit       >= 1M daily streams

High Performer >= 100K daily streams

Regular         < 100K

In [0]:
# Create Stream Tier
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "stream_tier",
        when(col("streams") >= 10000000, "Mega Hit")
        .when(col("streams") >= 1000000, "Super Hit")
        .when(col("streams") >= 100000, "High Performer")
        .otherwise("Regular")
    )
)

In [0]:
stream_tier_check = (
    silver_song_charts
    .groupBy("stream_tier")
    .agg(
        count("*").alias("records"),
        countDistinct("uri").alias("unique_songs"),
        sum("streams").alias("total_streams")
    )
    .orderBy(desc("records"))
)

stream_tier_check.show()

+--------------+--------+------------+-------------+
|   stream_tier| records|unique_songs|total_streams|
+--------------+--------+------------+-------------+
|       Regular|37491874|      233057| 735932652380|
|High Performer| 4759061|       63725|1241708146351|
|     Super Hit|  505314|        9001| 914150952309|
|      Mega Hit|     577|         117|   7121019474|
+--------------+--------+------------+-------------+



In [0]:
silver_song_charts.printSchema()

root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- rank: long (nullable = true)
 |-- uri: string (nullable = true)
 |-- artist_names: string (nullable = false)
 |-- track_name: string (nullable = false)
 |-- label: string (nullable = false)
 |-- peak_rank: long (nullable = true)
 |-- previous_rank: long (nullable = true)
 |-- days_on_chart: long (nullable = true)
 |-- streams: long (nullable = true)
 |-- consecutive_days: long (nullable = true)
 |-- entry_status: string (nullable = true)
 |-- peak_date: date (nullable = true)
 |-- entry_rank: long (nullable = true)
 |-- entry_date: date (nullable = true)
 |-- release_date: date (nullable = true)
 |-- artist_uris: string (nullable = true)
 |-- valid_release_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- song_age_days: integer (nullable = true)
 |-- song_age_category: s

In [0]:
len(silver_song_charts.columns)

30

Label Standardization Check

In [0]:
label_quality_check = (
    silver_song_charts
    .select(
        countDistinct("label").alias("unique_labels"),
        count(
            when(
                col("label").isNull(),
                True
            )
        ).alias("null_labels"),
        count(
            when(
                trim(col("label")) == "",
                True
            )
        ).alias("empty_labels")
    )
)

label_quality_check.show()

+-------------+-----------+------------+
|unique_labels|null_labels|empty_labels|
+-------------+-----------+------------+
|        27948|          0|           0|
+-------------+-----------+------------+



In [0]:
top_labels = (
    silver_song_charts
    .groupBy("label")
    .agg(
        countDistinct("uri").alias("songs"),
        sum("streams").alias("streams")
    )
    .orderBy(desc("streams"))
)

top_labels.show(20,False)

+---------------------------+-----+------------+
|label                      |songs|streams     |
+---------------------------+-----+------------+
|Columbia                   |3333 |127540043438|
|Republic Records           |742  |80928579795 |
|Warner Records             |1133 |67241482344 |
|Atlantic Records           |1002 |61379915912 |
|Rimas Entertainment LLC    |499  |56247710351 |
|Taylor Swift               |375  |49017523462 |
|Sony Music Latin           |1282 |48186424558 |
|Atlantic Records UK        |624  |45674690037 |
|WEA Latina                 |1117 |40617848158 |
|Sony Music Entertainment   |4547 |37855830484 |
|Island Records             |389  |34753979909 |
|Darkroom/Interscope Records|84   |34542290457 |
|BIGHIT MUSIC               |662  |33947203626 |
|UMLE - Latino              |466  |33881884803 |
|RCA Records Label          |1612 |32187810992 |
|Som Livre                  |862  |27143871815 |
|Rimas Entertainment LLC.   |406  |26906454777 |
|T-Series           

In [0]:
label_case_check = (
    silver_song_charts
    .select(
        lower(trim(col("label"))).alias("clean_label"),
        col("label")
    )
    .groupBy("clean_label")
    .agg(
        countDistinct("label").alias("variations")
    )
    .filter(col("variations") > 1)
    .orderBy(desc("variations"))
)

label_case_check.show(20, False)

+-----------------------------+----------+
|clean_label                  |variations|
+-----------------------------+----------+
|vla music entertainment      |4         |
|hits records                 |3         |
|kreisais krasts              |3         |
|22 records                   |3         |
|kyd records                  |3         |
|master media ltda            |3         |
|starship entertainment       |3         |
|dollar tv                    |3         |
|kosso                        |3         |
|od srdca pre love            |3         |
|pledis entertainment         |3         |
|bandoscope records           |3         |
|independent                  |3         |
|treasure box                 |3         |
|musica                       |3         |
|igroovemusic.com             |3         |
|hpf music                    |3         |
|karma company                |3         |
|königsrasse                  |3         |
|nmg/g-huset & kingpin record$|3         |
+----------

Create Standardized Label & artists

In [0]:
silver_song_charts = (
    silver_song_charts
    .withColumn(
        "standardized_label",
        initcap(
            lower(
                trim(col("label"))
            )
        )
    )
)

In [0]:
label_case_check = (
    silver_song_charts
    .select(
        lower(trim(col("standardized_label"))).alias("clean_label"),
        col("standardized_label")
    )
    .groupBy("clean_label")
    .agg(
        countDistinct("standardized_label").alias("variations")
    )
    .filter(col("variations") > 1)
)

label_case_check.show()

+-----------+----------+
|clean_label|variations|
+-----------+----------+
+-----------+----------+



In [0]:
silver_artist_mapping = (
    silver_artist_mapping
    .withColumn(
        "standardized_artist_name",
        initcap(
            lower(
                trim(col("artist_name"))
            )
        )
    )
)

In [0]:
silver_artist_mapping.printSchema()

root
 |-- uri: string (nullable = true)
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- standardized_artist_name: string (nullable = true)



In [0]:
artist_case_check = (
    silver_artist_mapping
    .select(
        lower(trim(col("standardized_artist_name"))).alias("clean_artist"),
        col("standardized_artist_name")
    )
    .groupBy("clean_artist")
    .agg(
        countDistinct("standardized_artist_name").alias("variations")
    )
    .filter(col("variations") > 1)
)

artist_case_check.show()

+------------+----------+
|clean_artist|variations|
+------------+----------+
+------------+----------+



In [0]:
print("Silver Song Columns:", len(silver_song_charts.columns))
print("Artist Mapping Columns:", len(silver_artist_mapping.columns))

print("Silver Song Records:")
print(silver_song_charts.count())

print("Artist Mapping Records:")
print(silver_artist_mapping.count())

Silver Song Columns: 31
Artist Mapping Columns: 4
Silver Song Records:
42756826
Artist Mapping Records:
379930
